# Quantitative Stock Market Analysis & Portfolio Simulator
### Repository Portfolio Project: `Stock-Market-Analysis-Python`

This notebook demonstrates professional data science workflows applied to quantitative finance. We will cover:
1. **Data Ingestion** from Yahoo Finance API.
2. **Data Cleaning & Sanitation** (formatting, NaN filling, duplication controls).
3. **Financial Statistical Analysis** (Volatility, CAGR, Sharpe Ratio, Peak-to-Trough Maximum Drawdowns).
4. **Visual Explorations** (Moving averages, distributions, Pearson correlation grids).
5. **Weighted Portfolio Optimization** vs S&P 500 Benchmarks.

## 1. Environment Setup & Library Imports

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# Set matplotlib aesthetic properties
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
print("Environment initialized successfully!")

## 2. Dynamic Ingestion and Sanitation Module

In [ ]:
tickers = ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'SPY']
start = '2025-01-01'
end = '2025-12-31'

print(f"Downloading {tickers} from {start} to {end}...")
raw_data = yf.download(tickers, start=start, end=end, group_by='ticker')

# Data Ingestion Sanitation checks
print("Raw shape:", raw_data.shape)

# Check NaNs
nan_count = raw_data.isnull().sum().sum()
if nan_count > 0:
    print(f"Sanitizing {nan_count} missing entries via Forward Fill...")
    raw_data = raw_data.ffill().bfill()
else:
    print("No missing values detected.")

## 3. Financial Metric Engine & Indicator Formulations
We calculate the following core quantitative structures:
1. **Daily Returns**:
   $$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$
2. **Compound Annualized Growth Rate (CAGR)**:
   $$\text{Annualized Return} = \left( \frac{P_{\text{final}}}{P_{\text{initial}}} \right)^{\frac{252}{N}} - 1$$
3. **Annualized Volatility**:
   $$\sigma_{\text{annual}} = \text{std}(R_t) \times \sqrt{252}$$
4. **Sharpe Ratio** (given risk-free rate $R_f = 2\%$):
   $$\text{Sharpe} = \frac{\text{Annualized Return} - R_f}{\sigma_{\text{annual}}}$$

In [ ]:
def calculate_metrics(price_series, risk_free_rate=0.02):
    daily_returns = price_series.pct_change().dropna()
    cum_return = (price_series.iloc[-1] / price_series.iloc[0]) - 1.0
    
    # Annualization factor
    n_years = len(price_series) / 252.0
    ann_return = (price_series.iloc[-1] / price_series.iloc[0]) ** (1.0 / n_years) - 1.0
    
    ann_vol = daily_returns.std() * np.sqrt(252)
    sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else 0
    
    # Maximum Drawdown
    peaks = price_series.cummax()
    drawdowns = (price_series - peaks) / peaks
    max_dd = drawdowns.min()
    
    return {
        'Cumulative Return': cum_return,
        'Annualized Return (CAGR)': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd
    }

# Sample calculation for AAPL
aapl_close = raw_data['AAPL']['Close']
aapl_metrics = calculate_metrics(aapl_close)
for metric, value in aapl_metrics.items():
    print(f"{metric}: {value*100:.2f}%" if '%' in metric or 'Return' in metric or 'Volatility' in metric or 'Drawdown' in metric else f"{metric}: {value:.2f}")

## 4. Visualizing Price Series & Moving Averages

In [ ]:
# Simple and exponential moving averages mapping
sma_50 = aapl_close.rolling(window=50).mean()
sma_200 = aapl_close.rolling(window=200).mean()

plt.figure(figsize=(14, 7))
plt.plot(aapl_close.index, aapl_close.values, label='AAPL Price', color='#1f77b4', linewidth=2)
plt.plot(sma_50.index, sma_50.values, label='50-day SMA', color='#ff7f0e', linestyle='--')
plt.plot(sma_200.index, sma_200.values, label='200-day SMA', color='#2ca02c', linestyle='-.')
plt.title('Apple Inc. Price Curves with SMAs Overlay', fontsize=14, fontweight='bold')
plt.ylabel('Price (USD)')
plt.xlabel('Date')
plt.legend(loc='upper left')
plt.show()

## 5. Cross-Asset Risk Correlation Heatmap

In [ ]:
# Construct returns df
returns_df = pd.DataFrame()
for t in ['AAPL', 'MSFT', 'NVDA', 'TSLA']:
    returns_df[t] = raw_data[t]['Close'].pct_change()

corr_matrix = returns_df.dropna().corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Asset Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Weighted Portfolio Simulation vs S&P 500 (SPY)
We model an equal-weighted portfolio allocating 25% each to AAPL, MSFT, NVDA, and TSLA.

In [ ]:
weights = {'AAPL': 0.25, 'MSFT': 0.25, 'NVDA': 0.25, 'TSLA': 0.25}
capital = 10000.0

# Normalize stock curves
normalized_df = pd.DataFrame()
for t, w in weights.items():
    normalized_df[t] = (raw_data[t]['Close'] / raw_data[t]['Close'].iloc[0]) * w

portfolio_equity = normalized_df.sum(axis=1) * capital
spy_equity = (raw_data['SPY']['Close'] / raw_data['SPY']['Close'].iloc[0]) * capital

plt.figure(figsize=(14, 7))
plt.plot(portfolio_equity.index, portfolio_equity.values, label='Equally-Weighted Portfolio', color='#2ca02c', linewidth=2.5)
plt.plot(spy_equity.index, spy_equity.values, label='S&P 500 Benchmark (SPY)', color='#7f7f7f', linestyle=':')
plt.title('Historical Equity Curve vs Benchmark', fontsize=14, fontweight='bold')
plt.ylabel('Portfolio Valuation (USD)')
plt.xlabel('Date')
plt.legend(loc='upper left')
plt.show()